# Two-Stage Retrieval Benchmark (NRMS)

This notebook benchmarks the **retrieval stage** that feeds the trained **NRMS** reranker in a two-stage news recommendation pipeline.

## Pipeline
1. **Retrieval** — given a user's clicked *history*, retrieve a candidate set `K` from the full news corpus using one of:
   - `tfidf` — TF-IDF (title + abstract), cosine similarity
   - `dense` — `sentence-transformers/all-MiniLM-L6-v2` mean-pooled embeddings
   - `entity` — Wikidata entity embeddings (knowledge-graph signal)
2. **Reranking** — the retrieved set is scored by the trained NRMS model via the existing `src/train.py::evaluate` (impression AUC / MRR / nDCG@5 / nDCG@10).

## Scale
- **Smoke test (this notebook, default):** small corpus slice + few dev impressions to verify the pipeline runs end-to-end without OOM.
- **Full run (big system):** set `MAX_NEWS = None` and `MAX_IMPRESSIONS = None` to index the full ~65k corpus.

In [ ]:
import os
import sys
import json

import pandas as pd
import torch

sys.path.insert(0, '.')

from src.data import prepare_data
from src.model import build_default_nrms
from src.train import load_checkpoint, get_device
from src.retrieval_eval import run_benchmark, results_to_dataframe

## 1. Run Configuration

Set the smoke-test scale here. For the full run on a big system, set `MAX_NEWS = None` and `MAX_IMPRESSIONS = None`.

In [ ]:
DATA_TRAIN = 'data/MINDsmall_train'
DATA_DEV = 'data/MINDsmall_dev'
ENTITY_VEC = os.path.join(DATA_TRAIN, 'entity_embedding.vec')
CKPT = 'checkpoints/runs_1784137259/best_model.pt'
CFG = 'checkpoints/runs_1784137259/run_config.json'

# --- Smoke-test scale (small). Set to None for the full run on a big system. ---
MAX_NEWS = 2000          # cap on indexed corpus size (head of combined train+dev news)
MAX_IMPRESSIONS = 30     # number of dev impressions to evaluate
K = 50                   # retrieved candidate set size (matches --max_candidates)
EVAL_BATCH_SIZE = 8

# Toggle retrievers
USE_TFIDF = True
USE_DENSE = True
USE_ENTITY = True

## 2. Load trained NRMS checkpoint

Rebuild the model exactly as during training (vocab, embed_dim, buffer) so the checkpoint loads cleanly, then run the retrieval benchmark.

In [ ]:
with open(CFG) as f:
    a = json.load(f)['args']
device = get_device()

(train_ds, in_time_ds, dev_ds, vocab, ntt, num_news, n2i, i2c, i2s, nc, ns) = prepare_data(
    train_behaviors_path=os.path.join(DATA_TRAIN, 'behaviors.tsv'),
    train_news_path=os.path.join(DATA_TRAIN, 'news.tsv'),
    dev_behaviors_path=os.path.join(DATA_DEV, 'behaviors.tsv'),
    dev_news_path=os.path.join(DATA_DEV, 'news.tsv'),
    max_history_len=a['max_history_len'],
    max_title_len=a['max_title_len'],
    min_word_freq=a['min_word_freq'],
    max_train_impressions=a['max_train_impressions'],
    max_dev_impressions=a['max_dev_impressions'],
    train_mode=a['train_mode'],
    max_candidates=a['max_candidates'],
    seed=a['seed'],
)

model = build_default_nrms(
    vocab_size=len(vocab),
    word_embed_dim=a['embed_dim'],
    num_heads=a['num_heads'],
    user_num_heads=a['user_num_heads'],
    max_title_len=a['max_title_len'],
    dropout=a['dropout'],
    category_mode=a['category_mode'],
    num_categories=nc,
    num_subcategories=ns,
    cat_embed_dim=a['cat_embed_dim'],
    subcat_embed_dim=a['subcat_embed_dim'],
)
model.set_news_title_tokens(ntt)
load_checkpoint(model, CKPT, device)
model.to(device)
model.eval()
criterion = torch.nn.BCEWithLogitsLoss()
print("Model loaded on", device)

## 3. Execute benchmark (retrieval -> NRMS rerank)

`run_benchmark` builds the corpus index once, then evaluates each enabled retriever end-to-end.

In [ ]:
results = run_benchmark(
    train_news_path=os.path.join(DATA_TRAIN, 'news.tsv'),
    dev_news_path=os.path.join(DATA_DEV, 'news.tsv'),
    dev_behaviors_path=os.path.join(DATA_DEV, 'behaviors.tsv'),
    entity_vec_path=ENTITY_VEC,
    model=model,
    device=device,
    criterion=criterion,
    k=K,
    max_impressions=MAX_IMPRESSIONS,
    eval_batch_size=EVAL_BATCH_SIZE,
    use_tfidf=USE_TFIDF,
    use_dense=USE_DENSE,
    use_entity=USE_ENTITY,
    max_news=MAX_NEWS,
)

## 4. Results

Side-by-side comparison of the three retrievers.

- `recall@k` / `hit_rate` — standalone retrieval quality (fraction of clicked news found in top-K).
- `retrieved_*` — end-to-end metrics after NRMS reranks the retrieved set.

> Note: on a smoke-test corpus slice the clicked positives usually live outside the small index, so `recall@k` is ~0 by design. The full run recovers meaningful recall.

In [ ]:
df = results_to_dataframe(results)
pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 30)
print(f"=== SMOKE TEST (max_news={MAX_NEWS}, max_impressions={MAX_IMPRESSIONS}) ===")
df

## 5. Full run on a big system

For the full-scale benchmark, change the config in cell 2:

```python
MAX_NEWS = None          # index the full ~65k corpus
MAX_IMPRESSIONS = None   # evaluate all dev impressions
EVAL_BATCH_SIZE = 64     # raise on GPU
```

Then re-run cells 2–4. The Dense retriever batches corpus embedding internally, so it will not OOM on the full corpus given enough RAM / a GPU.